# 📘 学习注释版：Silver ERP Customer

**Input：** `workspace.bronze.erp_cust_az12`  
**Output：** `workspace.silver.erp_customers`

最重要的业务点：

```text
NASAW00011000
→ 去掉 NAS
→ AW00011000
```

这样 ERP Customer Key 才能和 CRM Customer Key JOIN。


#Initialization

## 🧰 学习说明：导入 PySpark 函数/类型

这里仅准备后续清洗需要的 API，例如：
- `col()`：引用列
- `trim()`：去前后空格
- `StringType`：判断字符串类型
- `F.when()`：类似 SQL `CASE WHEN`

这一 Cell 不改变数据。


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Read Bronze table

## 📖 学习说明：读取 Bronze Table

**Input：** `workspace.bronze.erp_cust_az12`  
**Process：** `spark.table()` 把 Catalog 中的 Table 读取成 DataFrame  
**Output：** 变量 `df`

后续所有 Silver 清洗都在这个 DataFrame 上进行。


In [0]:
df = spark.table("workspace.bronze.erp_cust_az12")

#Silver Transformations

##Trimming

## ✂️ 学习说明：批量 Trim 字符串

遍历 DataFrame 所有字段：

```text
如果字段类型 = String
→ trim()
→ 去掉前后空格
```

真实数据中 `" Jon"` 和 `"Jon"` 在比较/JOIN 时可能被认为不同，所以 Silver 要先清理。


In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##Customer ID Cleanup

## 🔗 学习说明：统一 ERP Customer Key

ERP：

```text
NASAW00011000
```

CRM：

```text
AW00011000
```

去掉 `NAS` 后，两套系统才能使用同一个 Key JOIN。


In [0]:
df = df.withColumn(
    "cid",
    F.when(col("cid").startswith("NAS"),
           F.substring(col("cid"), 4, F.length(col("cid"))))
     .otherwise(col("cid"))
)


##Birthdate Validation

## 🎂 学习说明：Birth Date 合理性验证

如果生日大于今天：

```text
未来生日 → NULL
```

这是业务合理性（Data Quality）检查。


In [0]:
df = df.withColumn(
    "bdate",
    F.when(col("bdate") > F.current_date(), None)
     .otherwise(col("bdate"))
)

##Gender Normalization

## 👤 学习说明：ERP Gender 标准化

把多种可能写法统一：

```text
F / FEMALE → Female
M / MALE   → Male
其他       → n/a
```


In [0]:
df = df.withColumn(
    "gen",
    F.when(F.upper(col("gen")).isin("F", "FEMALE"), "Female")
     .when(F.upper(col("gen")).isin("M", "MALE"), "Male")
     .otherwise("n/a")
)

## Renaming Columns

## 🏷️ 学习说明：统一字段名称

把源系统字段名统一成业务更容易理解的名字。

例如：

```text
cst_id        → customer_id
cst_key       → customer_number
prd_nm        → product_name
```

意义：Silver 不只是“清洗值”，也统一数据模型/字段契约。


In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity checks of dataframe

## 👀 学习说明：DataFrame Sanity Check

只显示前 10 行，快速确认当前 DataFrame：
- 字段是否正确
- 清洗是否生效
- 数据是否仍然存在

这一步不写表，只是开发时的中间检查。


In [0]:
df.limit(10).display()

#Writing Silver Table

## 💾 学习说明：把 DataFrame 持久化为 Delta Table

**Input：** 当前 `df`  
**Process：**
- `mode("overwrite")`：目标已存在时覆盖
- `format("delta")`：使用 Delta 格式
- `saveAsTable()`：注册为 Catalog Table

**Output：** `workspace.silver.erp_customers`

注意：这也是为什么 Bootcamp 可以重复运行而通常不会因为“表已存在”直接失败。


In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customers")

## Sanity checks of silver table

## ✅ 学习说明：验证 ERP Customer Silver 表

重点看 `customer_number`：
- `NAS` 前缀是否已经删除
- 是否能与 CRM Customer Key 使用相同格式


In [0]:
%sql
SELECT * FROM workspace.silver.erp_customers LIMIT 10